In [1]:
################################ Read and match spectroscopy classification with S1 and S2 ####
#### Author: Boya ("Paul") Zhang

import numpy as np
from matplotlib import pyplot as plt

import shapefile as shp
from shapely.geometry import Point
from shapely.geometry import shape 
import pandas as pd
import matplotlib
import os
from datetime import datetime, timedelta
#import mpld3
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from scipy.stats import linregress
import matplotlib.font_manager as fm
from collections import Counter
import seaborn as sns
from osgeo import gdal
import subprocess
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd
from rasterio.mask import mask
from rasterio.plot import show
from osgeo import gdal, gdalconst, ogr
import glob
from scipy.ndimage import zoom
from rasterio.windows import Window
from rasterio.transform import Affine
from scipy import stats
from rasterio.merge import merge
from rasterio.sample import sample_gen

In [2]:
### mosaic the three warped images ############
path = 'D:/Paul/Groundwater/Spectroscopy/2021'
datasets = []

image_files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.tif')]

# Open each image file, read its data, and append the opened dataset to the list
for image_file in image_files:
    with rasterio.open(image_file) as src:
        datasets.append(src.read())



In [ ]:
mosaic, out_trans = merge(datasets)

# Define the output file path
output_file = path + '/mosaic_warp.tif'

# Write the mosaic to a new TIFF file
with rasterio.open(output_file, "w", driver='GTiff', width=mosaic.shape[2], height=mosaic.shape[1],
                   count=mosaic.shape[0], dtype=mosaic.dtype, crs=datasets[0].crs, transform=out_trans) as dest:
    dest.write(mosaic)

In [3]:
os.chdir(r'\\pinto\Winshare\Personal\Paul\Flooding\Spectroscopy\2021') 
folders = [folder for folder in os.listdir('.') if os.path.isdir(os.path.join('.', folder))]

In [4]:
folders

['S2A_MSIL2A_20210718T184921_N0301_R113_T10SEH_20210718T225851.SAFE',
 'S2A_MSIL2A_20210718T184921_N0500_R113_T10SFG_20230214T010718.SAFE',
 'S2A_MSIL2A_20210718T184921_N0500_R113_T10SFH_20230214T010718.SAFE',
 'S2B_MSIL2A_20210716T185919_N0301_R013_T10SEH_20210716T214658.SAFE']

In [5]:
## for each tile, mask with a Delta shapefile to get a subset

bands = ['B02_20m.jp2', 'B03_20m.jp2', 'B8A_20m.jp2', 'B11_20m.jp2', 'B12_20m.jp2']
ct = 0
shapefile_path = r'//pinto/Winshare/Personal/Paul/Flooding/YoloShape/Delta.shp'
shapefile = gpd.read_file(shapefile_path)

for i in range(1):
    print('folder is')
    print(i)
    os.chdir(r'\\pinto\Winshare\Personal\Paul\Flooding\Spectroscopy\2021') 
    folders = [folder for folder in os.listdir('.') if os.path.isdir(os.path.join('.', folder))]
    constant_path = folders[i]

    # String variable containing the variable part of the path
    variable_part = '.\GRANULE'

    # Concatenate the constant part with the variable part
    full_path = os.path.join(constant_path, variable_part)

    # Change the current working directory
    os.chdir(full_path)
    subfolder = [folder for folder in os.listdir('.') if os.path.isdir(os.path.join('.', folder))]
    variable_part = 'IMG_DATA\R20m'
    full_path = os.path.join(subfolder[0], variable_part)
    os.chdir(full_path)

    files = os.listdir('.')

    # Iterate over each filename and find the desired file
   
    for bd in range(len(bands)): 
        print("current band is:")
        print(bd)
        target_file = None
        for file in files:
            if file.endswith(bands[bd]):
                target_file = file
                break

        if ct == 0: ## for the first folder and the first band
            with rasterio.open(target_file) as src:
                # Read the image data
                out_image, out_transform = mask(src, shapefile.geometry, crop=True)
                # Update the metadata for the cropped image
                out_meta = src.meta.copy()
                out_meta.update({'driver': 'GTiff',
                                 'height': out_image.shape[1],
                                 'width': out_image.shape[2],
                                 'transform': out_transform})

                # Get image metadata
                meta1 = out_meta
            x, y = out_image.shape[-2:]
            img_data_3d = np.empty((5, x, y), dtype=out_image.dtype) ### for each image, we extract 5 bands. 
            img_data_3d[ct, :, :] = out_image[0, :, :]  # Assuming it's a single-band grayscale image
        
        else:
            with rasterio.open(target_file) as src:
                # Read the image data
                 # Read the image data
                out_image, out_transform = mask(src, shapefile.geometry, crop=True)
                # Update the metadata for the cropped image
                img_data_3d[ct, :, :] = out_image

                # Get image metadata
#                 if i == 1:
#                     meta2 = src.meta
#                 else:
#                     meta3 = src.meta
        ct = ct + 1

folder is
0
current band is:
0
current band is:
1
current band is:
2
current band is:
3
current band is:
4


In [6]:
################## Calculating AWEI #############################
## equation is B+2.5*G-1.5*(NIR+SWIR1)-0.25*SWIR2

if i ==0 :
    awei1 = img_data_3d[0, :, :] / 10000 + 2.5 * img_data_3d[1, :, :] / 10000 - 1.5 * (img_data_3d[2, :, :] / 10000 + img_data_3d[3, :, :] / 10000) - 0.25 * img_data_3d[4, :, :] / 10000
elif i == 1:
    awei2 = img_data_3d[0, :, :] / 10000 + 2.5 * img_data_3d[1 , :, :] / 10000 - 1.5 * (img_data_3d[2 , :, :] / 10000 + img_data_3d[3 , :, :]) / 10000 - 0.25 * img_data_3d[4, :, :] / 10000
else:
    awei3 = img_data_3d[0, :, :] / 10000 + 2.5 * img_data_3d[1 , :, :] / 10000 - 1.5 * (img_data_3d[2 , :, :] / 10000 + img_data_3d[3 , :, :]) / 10000 - 0.25 * img_data_3d[4, :, :] / 10000

In [8]:
## warp spectroscopy classification into the s2 subset

os.chdir('D:\Paul\Groundwater\Spectroscopy')
with rasterio.open('Delta_202107_mosaic_v7.tif') as src:
        # Read the image data
        spc19 = src.read()
        meta19 = src.meta

################ rasterio warp spc19 to the subset of the first folder image ######
spc_warp = np.empty((spc19.shape[0], meta1['height'], meta1['width']), dtype=spc19.dtype)
reproject(
    source=spc19,
    destination=spc_warp,
    src_transform=meta19['transform'],
    src_crs=meta19['crs'],
    dst_transform=meta1['transform'],
    dst_crs=meta1['crs'],
    resampling=Resampling.nearest
)

meta_spcw = meta1.copy()
meta_spcw.update({
    'width': meta1['width'],
    'height': meta1['height']
})

In [10]:
pwd

'D:\\Paul\\Groundwater\\Spectroscopy'

In [12]:
## output
os.chdir(r'D:\Paul\Groundwater\Spectroscopy\2021')
with rasterio.open('spc_warp1.tif', 'w', **meta_spcw) as dst:
    # Get the number of bands in the raster data
    num_bands = spc_warp.shape[0]  # Assuming the first dimension represents bands

    # Write each band of the raster data separately
    for band_idx in range(num_bands):
        dst.write(spc_warp[band_idx, :, :], band_idx + 1)  # Add 1 to band_idx to match rasterio's 1-based indexing

In [13]:
with rasterio.open('awei1.tif', 'w', **meta1) as dst:
    # Get the number of bands in the raster data
    num_bands = 1  # Assuming the first dimension represents bands

    # Write each band of the raster data separately
    for band_idx in range(num_bands):
        awei1 = awei1.astype(float)
        dst.write(awei1, band_idx + 1)  # Add 1 to band_idx to match rasterio's 1-based indexing

In [ ]:
awei1

In [ ]:
awei1.shape

In [ ]:
# Get the affine transformation matrix from the metadata
transform = meta_spcw['transform']

# Get the height and width of the raster data
height, width = spc_warp.shape[1:]

# Initialize arrays to store pixel coordinates
x_coords = np.empty((height, width))
y_coords = np.empty((height, width))

# Calculate pixel coordinates for all pixels in the raster
for i in range(height):
    for j in range(width):
        # Calculate pixel center coordinates
        x_coords[i, j], y_coords[i, j] = rasterio.transform.xy(transform, i, j)

# Flatten the arrays to get 1D arrays of coordinates
x_coords_flat = x_coords.flatten()
y_coords_flat = y_coords.flatten()

In [ ]:
awei1_flat = awei1.flatten()
spc_warp_flat = spc_warp.flatten()

In [ ]:
mask = (spc_warp_flat == 14) | (spc_warp_flat == 10)

# Create a new array with values replaced according to the mask
flood = np.where(mask, 1, 0)

In [ ]:
os.chdir('D:\Paul\Groundwater\Spectroscopy')
ind = np.arange(0, spc_warp_flat.shape[0], 1)
out = np.vstack((ind, x_coords_flat, y_coords_flat, awei1_flat, spc_warp_flat, flood))
out = out.transpose()

with open('warp1.csv',"w") as f:
    np.savetxt(f, out, delimiter=",", header="ID, x, y, AWEI, classification, flood", 
               fmt="%f")

In [ ]:
df1 = pd.read_csv('D:/Paul/Groundwater/Spectroscopy/2019/warp1.csv')
df2 = pd.read_csv('D:/Paul/Groundwater/Spectroscopy/2019/warp2.csv')
df3 = pd.read_csv('D:/Paul/Groundwater/Spectroscopy/2019/warp3.csv')
merged_df = pd.concat([df1, df2, df3]).drop_duplicates(subset=[' x', ' y'])

# Reset the index of the merged DataFrame
merged_df.reset_index(drop=True, inplace=True)
merged_df.to_csv('D:/Paul/Groundwater/Spectroscopy/2019/2019.csv', index=False)  # Set index=False to avoid writing row numbers as a column

In [ ]:
spc_warp = np.empty((spc19.count, meta1['height'], meta1['width']), dtype=spc19.dtypes[0])


In [ ]:
df = pd.read_csv('D:/Paul/Groundwater/Spectroscopy/2019/2019.csv')


In [ ]:
zoomed_meta = meta19.copy()

# Update width and height based on the new shape of the zoomed raster
zoomed_meta['width'] = spc19_zoom.shape[1]
zoomed_meta['height'] = spc19_zoom.shape[0]

# Update the transformation matrix to reflect the changes in resolution
original_transform = meta19['transform']
new_resolution_x = original_transform.a / resampling_ratio
new_resolution_y = original_transform.e / resampling_ratio
new_transform = Affine(new_resolution_x, original_transform.b, original_transform.c,
                       original_transform.d, new_resolution_y, original_transform.f)

zoomed_meta['transform'] = new_transform

In [ ]:
extent_1 = rasterio.windows.bounds(Window(0, 0, meta1['width'], meta1['height']), meta1['transform'])
extent_2 = rasterio.windows.bounds(Window(0, 0, zoomed_meta['width'], zoomed_meta['height']), zoomed_meta['transform']) ## use the resampled, instead of the original one. 
# Determine overlapping extent
overlap_extent = [max(extent_1[0], extent_2[0]), 
                  max(extent_1[1], extent_2[1]),
                  min(extent_1[2], extent_2[2]),
                  min(extent_1[3], extent_2[3])]
window = rasterio.windows.from_bounds(*overlap_extent, transform=zoomed_meta['transform'])
subset_img_data_2 = spc19_zoom[:, int(window.row_off):int(window.row_off) + int(window.height), int(window.col_off):int(window.col_off) + int(window.width)]
subset_meta_data_2 = zoomed_meta.copy()
subset_meta_data_2['width'] = window.width
subset_meta_data_2['height'] = window.height
subset_meta_data_2['transform'] = rasterio.windows.transform(window, zoomed_meta['transform'])

In [ ]:
subset_img_data_2.shape

In [ ]:
overlap_extent

In [ ]:
extent_1 = rasterio.windows.bounds(meta1['width'], meta1['height'], meta1['transform'])
extent_2 = rasterio.windows.bounds(meta19['width'], meta19['height'], meta19['transform'])

# Determine overlapping extent
overlap_extent = [max(extent_1[0], extent_2[0]), 
                  max(extent_1[1], extent_2[1]),
                  min(extent_1[2], extent_2[2]),
                  min(extent_1[3], extent_2[3])]

# Convert overlapping extent to window
window = rasterio.windows.from_bounds(*overlap_extent, transform=meta_data_2['transform'])

# Read the subset of the second raster using the window
subset_img_data_2 = img_data_2[:, window.row_off:window.row_off + window.height, window.col_off:window.col_off + window.width]

# Update metadata for the subset raster
subset_meta_data_2 = meta_data_2.copy()
subset_meta_data_2['width'] = window.width
subset_meta_data_2['height'] = window.height
subset_meta_data_2['transform'] = rasterio.windows.transform(window, meta_data_2['transform'])

In [ ]:
img_data_3d[0, :, :].shape

In [ ]:
spc19.shape



In [ ]:
meta19

In [ ]:
folders

In [ ]:
plt.imshow(spc19_zoom[0], cmap='viridis', interpolation='nearest')

In [ ]:
plt.imshow(spc19_zoom[0, :, :], cmap='viridis')  # Assuming you want to plot the first 'slice

In [ ]:
plt.figure(figsize=(8, 8))
plt.hist(awei1.flatten(), bins=20, color='skyblue', edgecolor='black')
plt.title('Pixel Value Histogram')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()